# 03 · Train and evaluate the field-risk classifier on the supplied dataset

This version uses the actual files in `data/synthetic/`: the split CSVs provide labels and split membership; the matching baseline/drift payloads provide the raw field values required by `FieldRiskClassifier`. It does **not** feed target labels, scenario names, record IDs, group IDs, or raw CSV-derived target features directly into the model.

**Outputs:** `models/field_classifier.json`, `models/field_classifier.manifest.json`, `results/field_classifier_eval.json`.

**Important dataset limitation:** the supplied split is a record-level train/validation/test split. It does not define a dedicated `test_unseen` field-name split, so this notebook reports the held-out test set and new/renamed-field subsets without claiming all test field names are unseen.

In [ ]:
import csv, json, hashlib, platform, sys, time, collections
from pathlib import Path
import numpy as np

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "data" / "synthetic" / "field_classification_train.csv").exists()), HERE)
DATA = ROOT / "data" / "synthetic"
MODELS, RESULTS = ROOT / "models", ROOT / "results"
MODELS.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
try:
    from app.detectors import validators as V
    from app.detectors.field_classifier import ClassifierConfig, FieldObservation, FieldRiskClassifier
except ImportError:
    # Local development fallback when the detector files are beside the notebook.
    import validators as V
    from field_classifier import ClassifierConfig, FieldObservation, FieldRiskClassifier

# The current CSV contains engineered audit columns, while FieldRiskClassifier
# expects raw FieldObservation inputs. Rebuild each observation from the matching
# payload value and retain only labels/split/drift metadata from the CSV.
def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]
def flatten_values(obj, prefix=""):
    out = {}
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            out.update(flatten_values(value, path))
    elif isinstance(obj, list):
        for value in obj: out.update(flatten_values(value, prefix + "[]"))
    else:
        out[prefix] = obj
    return out
payload_rows = read_jsonl(DATA / "baseline_payloads.jsonl") + read_jsonl(DATA / "drift_payloads.jsonl")
payload_by_id = {r["record_id"]: r for r in payload_rows}

def drift_category(row):
    if int(row.get("is_new_field", 0)): return "NEW_FIELD"
    if int(row.get("is_renamed_field", 0)): return "RENAME_MATCH"
    if int(row.get("has_type_shift", 0)): return "TYPE_SHIFT"
    if int(row.get("has_distribution_shift", 0)): return "DISTRIBUTION_SHIFT"
    return "NO_DRIFT"

def load_field_split(split):
    path = DATA / f"field_classification_{split}.csv"
    with path.open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    result = []
    missing = []
    for row in rows:
        rec = payload_by_id.get(row["record_id"])
        if rec is None:
            missing.append(row["record_id"]); continue
        values = flatten_values(rec["payload"])
        field_path = row["field_path"]
        if field_path not in values:
            missing.append(f"{row['record_id']}:{field_path}"); continue
        value = values[field_path]
        obs = FieldObservation(field_path=field_path, values=[value], drift_category=drift_category(row))
        is_text = (row.get("field_name", "").lower() in {"notes", "message", "description", "comment", "clinical_note", "support_text"}
                   or (isinstance(value, str) and len(value) >= 48))
        result.append({"record_id": row["record_id"], "group_id": row.get("group_id", row["record_id"]),
                       "field_path": field_path, "field_name": row.get("field_name", field_path.split(".")[-1]),
                       "value": value, "observation": obs, "label": int(row["label_sensitive"]),
                       "drift_label": int(row["label_drift_anomaly"]), "drift_category": obs.drift_category,
                       "is_text": bool(is_text), "category": row.get("category", "") or "",
                       "is_new_field": int(row.get("is_new_field", 0)),
                       "is_renamed_field": int(row.get("is_renamed_field", 0)),
                       "has_type_shift": int(row.get("has_type_shift", 0)),
                       "has_distribution_shift": int(row.get("has_distribution_shift", 0))})
    if missing:
        raise ValueError(f"{split}: {len(missing)} field rows could not be matched to payload values; examples: {missing[:5]}")
    return result

splits = {s: load_field_split(s) for s in ("train", "validation", "test")}
print("Project root:", ROOT, "| machine:", platform.machine())
print("Field rows:", {s: len(v) for s, v in splits.items()})

## Metrics and validator baseline

False-negative rate (FNR) estimates the share of sensitive fields missed. The validator baseline uses whole-value matching for structured fields and embedded matching for likely free-text fields.

In [ ]:
def metrics(y, pred):
    y, pred = np.asarray(y, dtype=bool), np.asarray(pred, dtype=bool)
    tp = int((y & pred).sum()); fp = int((~y & pred).sum())
    fn = int((y & ~pred).sum()); tn = int((~y & ~pred).sum())
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    return {"n": len(y), "precision": p, "recall": r, "f1": 2*p*r/(p+r) if p+r else 0.0,
            "fnr": fn/(tp+fn) if tp+fn else 0.0, "fpr": fp/(fp+tn) if fp+tn else 0.0,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}

def y_of(split): return np.asarray([r["label"] for r in splits[split]], dtype=int)
def obs_of(split): return [r["observation"] for r in splits[split]]
def validator_score(row):
    vals = V.field_hit_ratios([row["value"]], embedded=row["is_text"])
    return max(vals.values(), default=0.0)

baseline = {s: np.asarray([validator_score(r) for r in splits[s]]) for s in ("validation", "test")}
for s in ("validation", "test"):
    print(s, "validator-only:", json.dumps(metrics(y_of(s), baseline[s] >= 0.40)))

## Train the production-compatible classifier

Only the training split is used to fit weights. Validation is used to choose an operating threshold; the test split is held out until final reporting.

In [ ]:
SEED = 13
cfg = ClassifierConfig(seed=SEED)
t0 = time.perf_counter()
clf = FieldRiskClassifier(cfg).fit(obs_of("train"), y_of("train"))
print(f"Training time: {time.perf_counter()-t0:.2f}s | model: {clf.model_version} | metadata: {clf.meta}")
proba = {s: clf.predict_proba(obs_of(s)) for s in ("validation", "test")}
for s in ("validation", "test"):
    print(s, "p>=0.50:", json.dumps(metrics(y_of(s), proba[s] >= 0.50)))
    print(s, "p>=0.70:", json.dumps(metrics(y_of(s), proba[s] >= 0.70)))

## Choose threshold on validation only

The selected threshold is the lowest threshold whose validation FPR stays within the configured budget. If no threshold satisfies it, the fallback is 0.50. This is an operating-point heuristic, not probability calibration.

In [ ]:
FPR_BUDGET = 0.05
sweep = []
for thr in np.round(np.arange(0.10, 0.96, 0.05), 2):
    sweep.append((float(thr), metrics(y_of("validation"), proba["validation"] >= thr)))
print(f"{'threshold':>10}{'precision':>11}{'recall':>9}{'F1':>8}{'FPR':>8}")
for thr, m in sweep:
    print(f"{thr:>10.2f}{m['precision']:>11.3f}{m['recall']:>9.3f}{m['f1']:>8.3f}{m['fpr']:>8.3f}")
valid_thresholds = [t for t, m in sweep if m["fpr"] <= FPR_BUDGET]
chosen = min(valid_thresholds) if valid_thresholds else 0.50
print("Chosen threshold:", chosen, "| validation FPR budget:", FPR_BUDGET)
print("Held-out test at chosen threshold:", json.dumps(metrics(y_of("test"), proba["test"] >= chosen)))

## Test performance by drift subset

This is a useful robustness slice, but `is_new_field`/`is_renamed_field` are generator metadata used only to group the evaluation results, not model features supplied as labels.

In [ ]:
test_rows = splits["test"]
subsets = {
    "all_test": list(range(len(test_rows))),
    "new_fields": [i for i, r in enumerate(test_rows) if r["is_new_field"] == 1],
    "renamed_fields": [i for i, r in enumerate(test_rows) if r["is_renamed_field"] == 1],
    "type_shift": [i for i, r in enumerate(test_rows) if r["has_type_shift"] == 1],
    "distribution_shift": [i for i, r in enumerate(test_rows) if r["has_distribution_shift"] == 1],
    "no_drift": [i for i, r in enumerate(test_rows) if r["drift_category"] == "NO_DRIFT"],
}
subset_metrics = {}
for name, idx in subsets.items():
    if not idx:
        subset_metrics[name] = {"n": 0, "note": "no rows"}; continue
    subset_metrics[name] = metrics(y_of("test")[idx], proba["test"][idx] >= chosen)
    print(name, json.dumps(subset_metrics[name]))

## Feature-group ablation

Retrain the same classifier with selected feature groups disabled. This tests the implementation's actual feature groups (name, value, drift, structure), not the precomputed CSV columns.

In [ ]:
variants = {
    "all_groups": {},
    "name_only": {"use_value": False, "use_drift": False, "use_structure": False},
    "value_only": {"use_name": False, "use_drift": False, "use_structure": False},
    "name_value_no_drift": {"use_drift": False},
    "value_drift": {"use_name": False},
    "name_drift": {"use_value": False},
}
ablation = {}
for name, flags in variants.items():
    model = clf if name == "all_groups" else FieldRiskClassifier(ClassifierConfig(seed=SEED, **flags)).fit(obs_of("train"), y_of("train"))
    p = model.predict_proba(obs_of("test"))
    ablation[name] = metrics(y_of("test"), p >= chosen)
    print(name, json.dumps(ablation[name]))

## Error analysis and explainability

Only paths and aggregate labels are printed; raw field values are not included in the saved report.

In [ ]:
test_pred = proba["test"] >= chosen
from collections import Counter
fn_by_name = Counter(r["field_name"] for r, y, p in zip(test_rows, y_of("test"), test_pred) if y == 1 and not p)
fp_by_name = Counter(r["field_name"] for r, y, p in zip(test_rows, y_of("test"), test_pred) if y == 0 and p)
print("False negatives by field name:", dict(fn_by_name.most_common(12)))
print("False positives by field name:", dict(fp_by_name.most_common(12)))
print("Largest numeric weights:")
for name, weight in sorted(clf.numeric_weights().items(), key=lambda kv: -abs(kv[1]))[:12]:
    print(f"  {name:<30} {weight:+.3f}")

## Save model and reproducibility report

In [ ]:
data_files = [DATA / f"field_classification_{s}.csv" for s in ("train", "validation", "test")]
data_sha = hashlib.sha256("".join(hashlib.sha256(p.read_bytes()).hexdigest() for p in data_files).encode()).hexdigest()
summary = {
    "model_version": clf.model_version, "seed": SEED, "data_sha256_combined": data_sha,
    "python": platform.python_version(), "machine": platform.machine(),
    "chosen_threshold": chosen, "validation_fpr_budget": FPR_BUDGET,
    "baseline_validator_only": {s: metrics(y_of(s), baseline[s] >= 0.40) for s in baseline},
    "full_model_p050": {s: metrics(y_of(s), proba[s] >= 0.50) for s in proba},
    "full_model_chosen": {s: metrics(y_of(s), proba[s] >= chosen) for s in proba},
    "test_subsets": subset_metrics, "ablation": ablation,
    "caveats": ["synthetic structured data", "record-level split; no dedicated unseen-name split", "s390x execution must be verified on LinuxONE"]
}
model_path = MODELS / "field_classifier.json"
clf.save(model_path)
reloaded = FieldRiskClassifier.load(model_path)
assert np.allclose(clf.predict_proba(obs_of("test")[:100]), reloaded.predict_proba(obs_of("test")[:100]))
(MODELS / "field_classifier.manifest.json").write_text(json.dumps({k: summary[k] for k in ("model_version", "seed", "data_sha256_combined", "python", "machine", "chosen_threshold", "caveats")}, indent=2))
(RESULTS / "field_classifier_eval.json").write_text(json.dumps(summary, indent=2))
print("Saved", model_path, "and", RESULTS / "field_classifier_eval.json", "| reload check passed")

In [ ]:
try:
    import matplotlib.pyplot as plt
    xs = [t for t, _ in sweep]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(xs, [m["precision"] for _, m in sweep], label="precision")
    ax.plot(xs, [m["recall"] for _, m in sweep], label="recall")
    ax.plot(xs, [m["fpr"] for _, m in sweep], label="FPR")
    ax.axvline(chosen, linestyle="--", color="grey", label="chosen")
    ax.set_xlabel("Threshold chosen using validation data"); ax.legend(); ax.set_title("Field classifier threshold sweep")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed; skipping optional chart")